In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active device: {device}")

In [ ]:
df = pd.read_csv('fashion-mnist_train.csv')
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2)

In [ ]:
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [ ]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size= 32, shuffle = True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle = False, pin_memory=True)

In [ ]:
class MyNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64, 10)
    )

  def forward(self, x):
    return self.model(x)


In [ ]:
learning_rate = 0.1
epochs = 100

In [ ]:
model = MyNN(X_train.shape[1])
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr= learning_rate, weight_decay=1e-4)

In [ ]:
for epoch in range(epochs):
  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()
        total_epoch_loss += loss.item()

  average_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch+1}, Loss: {average_loss}')




In [ ]:
model.eval()

In [16]:
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      output = model(batch_features)
      _, predicted = torch.max(output, 1)
      total += batch_labels.shape[0]

      correct += (predicted == batch_labels).sum().item()

print(f'Accuracy: {correct/total * 100:.2f}%')

Accuracy: 89.02%


In [17]:
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    output = model(batch_features)
    _, predicted = torch.max(output, 1)
    total += batch_labels.shape[0]

    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy: {correct/total * 100:.2f}%')

Accuracy: 94.30%
